# ARC-AGI-3 Solver — Qwen3.8-27B-FP8

This notebook runs the **TAAF ARC-AGI-3 solver** using a locally mounted **Qwen3.8-27B-FP8** checkpoint through an OpenAI-compatible **vLLM** inference server.

## Model

- **Model:** `Qwen/Qwen3.8-27B-FP8`
- **Format:** Hugging Face / Safetensors
- **Quantization:** FP8
- **Kaggle Model:** `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1`
- **Variation:** `hf-fp8`
- **Version:** `1`
- **Served model ID:** `Qwen/Qwen3.8-27B-FP8`

### Kaggle model path

```text
/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1

In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# Boot attestation (doctrine v2, 2026-08-17): the mounted weights must be the
# OFFICIAL Qwen3.8-FP8. Discriminators verified offline against both the
# official HF snapshot and the vrfai 3.6 config: 3.8 = quant_method fp8 /
# fmt e4m3 / transformers 5.8.0.dev0; 3.6-vrfai = compressed-tensors
# config_groups / transformers 5.6.2. A wrong mount must DIE here, before any
# game action is spent. --served-model-name is a rename and proves nothing.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
_q = _cfg.get("quantization_config") or {}
assert _cfg.get("architectures") == ["Qwen3_5ForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _q.get("quant_method") == "fp8" and _q.get("fmt") == "e4m3", (
    f"attest FAIL: quantization_config is not official fp8/e4m3: {_q}")
assert _cfg.get("transformers_version") == "5.8.0.dev0", (
    f"attest FAIL: transformers_version {_cfg.get('transformers_version')} "
    "(vrfai 3.6 stamps 5.6.2)")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_idx_path = QWEN_MODEL_PATH / "model.safetensors.index.json"
if _idx_path.is_file():
    print("attest: index sha256", _hashlib.sha256(_idx_path.read_bytes()).hexdigest())
_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
assert _shards, "attest FAIL: no safetensors shards at model path"
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert _total > 25_000_000_000, f"attest FAIL: total shard bytes {_total} too small for 27B FP8"
_h = _hashlib.sha256()
with open(_shards[0], "rb") as _f:
    _h.update(_f.read(1 << 20))
print("attest: first-shard-1MiB sha256", _h.hexdigest())

# Greedy decode fingerprint — logged (not asserted) for cross-run comparison.
_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": QWEN_SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=180) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — official Qwen3.8-FP8 signature verified before any game")


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Smoke/eval hook: a NORMAL COMMIT runs a 3-game, 60-min-soft-capped offline
# smoke (proves serve + attestation + agent loop on the scored GPU class).
# The scored rerun (KAGGLE_IS_COMPETITION_RERUN) never enters this branch —
# it plays the full competition games exactly as the upstream scaffold does.
SMOKE_GAMES = ["sb26-7fbdac44", "ft09-0d8bbf25", "tn36-ef4dde99"]

if not run_as_submission:
    import arc_agi
    from taaf.game_api import ArcadeSpec, GameAPI

    def _resolve_env_dir():
        candidates = [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"),
        ]
        for cand in candidates:
            if cand.is_dir():
                return str(cand)
        for hit in Path("/kaggle/input").rglob("environment_files"):
            if hit.is_dir():
                return str(hit)
        raise RuntimeError("environment_files dir not found in /kaggle/input")

    _env_dir = _resolve_env_dir()
    _spec = ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=_env_dir)
    bm.games = [GameAPI(env_name=name, arcade_spec=_spec) for name in SMOKE_GAMES]
    bm.n_passes = 1
    bm.game_weights = None
    bm.label = "duck38-v12-digest-smoke"
    soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=6300)
    print(f"smoke hook: {len(bm.games)} games, env_dir={_env_dir}, soft_end={soft_end}")
else:
    print("scored rerun: smoke hook inert — full competition games")

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))


In [ ]:
# ANIMATION DIGEST graft (single variable vs the v12 base): harness-side
# decoding of transient animation frames into <=60-token prompt text.
# Falsifiable proof (submission/_anim_digest/test_sb26_check.py, PASS): the
# digest reproduces in 24ms the scan-order/expected-color information that
# decided clear-vs-stuck on sb26 (801s of manual decoding vs a fatal misread).
# Seam-validated on 35/35 recorded animations. Fail-open; DIGEST=0 disables.
_DIGEST_SOURCE = '"""digest_animation: deterministic, compact (~60 token) narration of a multi-frame\nARC-AGI-3 action response, for injection into the duck harness action result.\n\nMotivation (depth study wf_9809fe85): sb26\'s check verdict exists only in transient\nanimation frames (a highlight visiting slots in sequence; final board unchanged).\nThe v12smoke run decoded it manually through ~10 LLM calls and 801s; banksmoke\nmisdecoded it and got stuck. This digest narrates the same structure in one line.\n\nStructure detected, in order of priority:\n  - blinks: consecutive distinct frames toggling the same cell set back and forth\n    (ft09 rejection flash, sb26 mismatch flash) -> "flash Rx3 @r0-39c20-28"\n  - reveals: a small region painted in place over several frames, ending in a\n    stable color (sb26 check pointer painting the expected color at each strip\n    slot, in visit order) -> "O@r3c10"\n  - movers: connected change-components tracked across frames by proximity,\n    rendered as direction-collapsed waypoint chains (sb26 slot ring diving from\n    the red tray to the green tray and back) -> "obj r21c14->r35c31->r17c31"\n  - fades: steps whose transitions are almost entirely +-1 grayscale ramps are\n    dropped (pure fade-in/fade-out carries no order information)\n  - scene changes: a step touching > SCENE_CHANGE_CELLS cells is reported as a\n    count, not narrated (e.g. the next level\'s board appearing after a WIN check)\n\nEvents are emitted in chronological order, so cross-track ordering (ring dove to\nthe green tray BETWEEN reveal 2 and reveal 3) survives into the text. That is\nexactly the slot-visit-order information sb26\'s check hides from the final frame.\n\nNo LLM, stdlib only, deterministic.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import Any, Iterable, Sequence\n\nCOLOR_CHARS = "WwgGcBMPRbSYOrNp"\n\nCLUSTER_GAP = 2            # chebyshev gap that still joins two changed cells\nTRACK_ATTACH_DIST = 16     # max centroid jump between consecutive comps of one mover\nMERGE_BBOX_AREA = 64       # comp merges into the previous waypoint if union bbox <= 8x8\nMERGE_CENTER_DIST = 2.0    # ...and its center stays put (a dwell, not a slow slide)\nFADE_DOMINANCE = 0.85      # fraction of +-1 grayscale transitions that makes a fade step\nSCENE_CHANGE_CELLS = 300   # bigger steps are reported as a count, not narrated\nMAX_EVENTS = 14            # hard budget on rendered events\n\nGrid = tuple[tuple[int, ...], ...]\n\n\n# --------------------------------------------------------------------------- utils\n\ndef _norm_grid(grid: Any) -> Grid:\n    rows = grid.tolist() if hasattr(grid, "tolist") else grid\n    return tuple(tuple(int(c) for c in row) for row in rows or ())\n\n\ndef _color(v: int | None) -> str:\n    if v is None:\n        return "?"\n    v = int(v)\n    return COLOR_CHARS[v] if 0 <= v < len(COLOR_CHARS) else "?"\n\n\ndef _diff(a: Grid, b: Grid, mask: frozenset) -> dict:\n    out = {}\n    for r in range(max(len(a), len(b))):\n        ra = a[r] if r < len(a) else ()\n        rb = b[r] if r < len(b) else ()\n        for c in range(max(len(ra), len(rb))):\n            va = ra[c] if c < len(ra) else None\n            vb = rb[c] if c < len(rb) else None\n            if va != vb and (r, c) not in mask:\n                out[(r, c)] = (va, vb)\n    return out\n\n\ndef _components(cells: Iterable[tuple[int, int]], gap: int = CLUSTER_GAP) -> list[list[tuple[int, int]]]:\n    """Cluster cells with chebyshev distance <= gap into connected components."""\n    todo = set(cells)\n    offsets = [(dr, dc) for dr in range(-gap, gap + 1) for dc in range(-gap, gap + 1) if dr or dc]\n    comps = []\n    while todo:\n        seed = min(todo)\n        todo.discard(seed)\n        comp, queue = [seed], [seed]\n        while queue:\n            r, c = queue.pop()\n            for dr, dc in offsets:\n                n = (r + dr, c + dc)\n                if n in todo:\n                    todo.discard(n)\n                    comp.append(n)\n                    queue.append(n)\n        comps.append(sorted(comp))\n    comps.sort(key=lambda comp: comp[0])\n    return comps\n\n\ndef _bbox(cells: Iterable[tuple[int, int]]) -> tuple[int, int, int, int]:\n    rows = [c[0] for c in cells]\n    cols = [c[1] for c in cells]\n    return min(rows), min(cols), max(rows), max(cols)\n\n\ndef _centroid(cells) -> tuple[float, float]:\n    rows = [c[0] for c in cells]\n    cols = [c[1] for c in cells]\n    return sum(rows) / len(rows), sum(cols) / len(cols)\n\n\ndef _pos_text(pos: tuple[float, float]) -> str:\n    return f"r{round(pos[0])}c{round(pos[1])}"\n\n\ndef _is_fade_step(census: Counter) -> bool:\n    total = sum(census.values())\n    if not total:\n        return False\n    fade = sum(\n        n for (old, new), n in census.items()\n        if old is not None and new is not None and abs(int(old) - int(new)) == 1\n        and max(int(old), int(new)) <= 5\n    )\n    return fade / total >= FADE_DOMINANCE\n\n\ndef _inverse(census: Counter) -> Counter:\n    return Counter({(new, old): n for (old, new), n in census.items()})\n\n\n# --------------------------------------------------------------------------- main\n\ndef digest_animation(\n    frames: Sequence[Any],\n    hud_mask_cells: Iterable[tuple[int, int]] | None = None,\n    before: Any | None = None,\n) -> str:\n    """Render a multi-frame action response as one compact deterministic line.\n\n    ``frames``: the engine\'s full frame list for one action (grids or ndarrays).\n    ``hud_mask_cells``: (row, col) cells to ignore entirely (HUD bars/timers).\n    ``before``: the board before the action, if available; sharpens step 1 and\n    enables net-change reporting.\n    Returns "" for single-frame responses.\n    """\n    grids = [_norm_grid(f) for f in frames or ()]\n    if len(grids) <= 1:\n        return ""\n    mask = frozenset(tuple(c) for c in hud_mask_cells or ())\n\n    # Collapse identical consecutive frames.\n    distinct: list[Grid] = []\n    for g in grids:\n        if not distinct or distinct[-1] != g:\n            distinct.append(g)\n\n    start = _norm_grid(before) if before is not None else distinct[0]\n    final = distinct[-1]\n\n    steps = []\n    prev = start\n    for i, g in enumerate(distinct):\n        d = _diff(prev, g, mask)\n        prev = g\n        if d:\n            steps.append({"i": i, "cells": d, "census": Counter(d.values())})\n\n    events: list = []  # (step, text, kind, track_id)\n\n    # --- blink pass: runs of identical cell sets with inverse transitions\n    consumed = set()\n    j = 0\n    while j < len(steps):\n        run = [j]\n        while (\n            run[-1] + 1 < len(steps)\n            and steps[run[-1] + 1]["i"] == steps[run[-1]]["i"] + 1\n            and set(steps[run[-1] + 1]["cells"]) == set(steps[run[-1]]["cells"])\n            and steps[run[-1] + 1]["census"] == _inverse(steps[run[-1]]["census"])\n        ):\n            run.append(run[-1] + 1)\n        if len(run) >= 2:\n            first = steps[run[0]]\n            on_color = first["census"].most_common(1)[0][0][1]\n            cycles = (len(run) + 1) // 2\n            r0, c0, r1, c1 = _bbox(first["cells"])\n            events.append((first["i"], f"flash {_color(on_color)}x{cycles} @r{r0}-{r1}c{c0}-{c1}", "blink", -1))\n            consumed.update(run)\n            j = run[-1] + 1\n        else:\n            j += 1\n    steps = [s for k, s in enumerate(steps) if k not in consumed]\n\n    # --- fade + scene-change pass\n    kept = []\n    for s in steps:\n        if _is_fade_step(s["census"]):\n            continue\n        if len(s["cells"]) > SCENE_CHANGE_CELLS:\n            events.append((s["i"], f"then {len(s[\'cells\'])}px scene change", "scene", -1))\n            continue\n        kept.append(s)\n    steps = kept\n\n    # --- track pass: cluster each step, associate comps to movers by proximity.\n    # All comps of one step that land on the same track are fused into a single\n    # observation (a sliding ring shows up as separate erase/draw edges).\n    tracks: list[dict] = []\n    for s in steps:\n        assignments: dict[int, list] = {}\n        for comp in _components(s["cells"]):\n            cen = _centroid(comp)\n            best, best_d = None, None\n            for ti, t in enumerate(tracks):\n                d = max(abs(t["last"][0] - cen[0]), abs(t["last"][1] - cen[1]))\n                if d <= TRACK_ATTACH_DIST and (best_d is None or d < best_d):\n                    best, best_d = ti, d\n            if best is None:\n                if len(comp) == 1:\n                    continue  # isolated single-cell speck (HUD tick)\n                tracks.append({"waypoints": [], "last": cen, "last_step": s["i"]})\n                best = len(tracks) - 1\n            assignments.setdefault(best, []).append(comp)\n        for ti in sorted(assignments):\n            t = tracks[ti]\n            cells = [cell for comp in assignments[ti] for cell in comp]\n            b = _bbox(cells)\n            cen = _centroid(cells)\n            wps = t["waypoints"]\n            merged = False\n            if wps:\n                wp = wps[-1]\n                u = (min(wp["bbox"][0], b[0]), min(wp["bbox"][1], b[1]),\n                     max(wp["bbox"][2], b[2]), max(wp["bbox"][3], b[3]))\n                center_shift = max(abs(cen[0] - wp["pos"][0]), abs(cen[1] - wp["pos"][1]))\n                if (\n                    (u[2] - u[0] + 1) * (u[3] - u[1] + 1) <= MERGE_BBOX_AREA\n                    and center_shift <= MERGE_CENTER_DIST\n                    and s["i"] - wp["last_step"] <= 3\n                ):\n                    wp["bbox"] = u\n                    wp["last_step"] = s["i"]\n                    wp["n_steps"] += 1\n                    wp["pos"] = ((u[0] + u[2]) / 2, (u[1] + u[3]) / 2)\n                    merged = True\n            if not merged:\n                wps.append({\n                    "pos": cen, "bbox": b, "first_step": s["i"], "last_step": s["i"],\n                    "n_steps": 1, "size": len(cells),\n                })\n            t["last"] = wps[-1]["pos"]\n            t["last_step"] = s["i"]\n\n    # --- reveal detection: a dwell whose bbox ends uniformly painted in a color\n    # it did not start with. Read from the actual frame, not the diff census, so\n    # a stray ring edge merged into the dwell cannot corrupt the verdict.\n    def _region_mode(grid: Grid, bbox: tuple[int, int, int, int]):\n        vals = [\n            grid[r][c]\n            for r in range(bbox[0], bbox[2] + 1)\n            for c in range(bbox[1], bbox[3] + 1)\n            if r < len(grid) and c < len(grid[r])\n        ]\n        if not vals:\n            return None, 0.0\n        v, n = Counter(vals).most_common(1)[0]\n        return v, n / len(vals)\n\n    for t in tracks:\n        for wp in t["waypoints"]:\n            wp["reveal"] = None\n            area = (wp["bbox"][2] - wp["bbox"][0] + 1) * (wp["bbox"][3] - wp["bbox"][1] + 1)\n            if wp["n_steps"] >= 2 and area <= MERGE_BBOX_AREA:\n                v, share = _region_mode(distinct[wp["last_step"]], wp["bbox"])\n                if v is not None and share >= 0.6:\n                    sv, sshare = _region_mode(start, wp["bbox"])\n                    if not (sv == v and sshare >= 0.6):\n                        wp["reveal"] = int(v)\n        # a dwell often erases (to white) then paints: keep only the later verdict\n        reveals = [wp for wp in t["waypoints"] if wp["reveal"] is not None]\n        for a, b in zip(reveals, reveals[1:]):\n            if max(abs(a["pos"][0] - b["pos"][0]), abs(a["pos"][1] - b["pos"][1])) <= 2:\n                a["reveal"] = None\n\n    # --- emit reveal events; emit motion segments for tracks without reveals.\n    # events: (step, text, kind, track_id); kind "arrow" is the most expendable.\n    for tid, t in enumerate(tracks):\n        wps = t["waypoints"]\n        has_reveal = any(wp["reveal"] is not None for wp in wps)\n        if has_reveal:\n            for wp in wps:\n                if wp["reveal"] is not None:\n                    events.append(\n                        (wp["first_step"], f"{_color(wp[\'reveal\'])}@{_pos_text(wp[\'pos\'])}", "reveal", tid)\n                    )\n            continue\n        # direction-collapse the waypoint chain\n        keep = [wps[0]]\n        last_dir = None\n        for wp in wps[1:]:\n            dr = wp["pos"][0] - keep[-1]["pos"][0]\n            dc = wp["pos"][1] - keep[-1]["pos"][1]\n            direction = ((dr > 1) - (dr < -1), (dc > 1) - (dc < -1))\n            if direction == last_dir and direction != (0, 0):\n                keep[-1] = wp  # extend the segment\n            else:\n                keep.append(wp)\n                last_dir = direction\n        if len(keep) == 1:\n            wp = keep[0]\n            events.append((wp["first_step"], f"chg {wp[\'size\']}px @{_pos_text(wp[\'pos\'])}", "chg", tid))\n        else:\n            events.append((keep[0]["first_step"], f"obj {_pos_text(keep[0][\'pos\'])}", "obj", tid))\n            for wp in keep[1:]:\n                events.append((wp["last_step"], f"->{_pos_text(wp[\'pos\'])}", "arrow", tid))\n\n    events.sort(key=lambda e: e[0])\n\n    # --- graded budget: drop intermediate motion waypoints first (keeping each\n    # track\'s final position), so reveals/blinks -- the order information --\n    # survive; only then elide the middle generically.\n    if len(events) > MAX_EVENTS:\n        last_arrow = {}\n        for k, e in enumerate(events):\n            if e[2] == "arrow":\n                last_arrow[e[3]] = k\n        droppable = [k for k, e in enumerate(events) if e[2] == "arrow" and last_arrow[e[3]] != k]\n        while len(events) > MAX_EVENTS and droppable:\n            mid = len(droppable) // 2\n            k = droppable.pop(mid)\n            events[k] = None\n            events = [e for e in events if e is not None]\n            last_arrow = {}\n            for k, e in enumerate(events):\n                if e[2] == "arrow":\n                    last_arrow[e[3]] = k\n            droppable = [k for k, e in enumerate(events) if e[2] == "arrow" and last_arrow[e[3]] != k]\n    if len(events) > MAX_EVENTS:\n        head = events[: MAX_EVENTS - 3]\n        tail = events[-2:]\n        omitted = len(events) - len(head) - len(tail)\n        events = head + [(head[-1][0], f"..{omitted} more..", "elide", -1)] + tail\n\n    # --- header + net change\n    header = f"{len(grids)}f anim"\n    if start == final:\n        header += " (board reverts)"\n\n    parts = [e[1] for e in events]\n    text = header + ": " + "; ".join(parts) if parts else header + ": no non-fade changes"\n\n    if before is not None:\n        net = _diff(_norm_grid(before), final, mask)\n        if not net:\n            text += "; no net change"\n        elif len(net) <= 8:\n            cen = _centroid(list(net))\n            colors = Counter(v[1] for v in net.values())\n            text += f"; net {len(net)}px {_color(colors.most_common(1)[0][0])}@{_pos_text(cen)}"\n        else:\n            r0, c0, r1, c1 = _bbox(list(net))\n            text += f"; net {len(net)}px r{r0}-{r1}c{c0}-{c1}"\n    return text\n'
_GRAFT_SOURCE = '"""Animation-digest graft — harness-side decoding of transient animation\nframes into compact prompt text.\n\nMotivation (measured, wf_9809fe85 + prototype test 2026-08-20): on sb26 the\nlevel-deciding information lives ONLY in transient animation frames; the run\nthat decoded them manually (801s, 10 LLM calls) cleared the level, its twin\nmisread them and got permanently stuck. digest_animation reproduces the\nrule-deciding content deterministically in ~24ms / ~30 tokens (falsifiable\ntest in submission/_anim_digest/test_sb26_check.py: PASS on both the failing\nand passing check scenarios).\n\nSeams (verified against the Aug-07 bundle):\n- inference/framework/solver.py:52 imports summarize_animation BY NAME and\n  calls it at :822 with the raw frames — we wrap the function and rebind it\n  in BOTH namespaces (the animation module AND solver\'s import-time binding;\n  the duck-mem silent-no-op lesson).\n- inference/agent/tool_agent.py:39 imports describe_animation BY NAME, calls\n  it at :1441 — same double rebind; the wrapper appends summary["digest"].\n\nFail-open: any exception inside either wrapper returns the stock result.\nDIGEST=0 disables at call time.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any\n\nfrom digest import digest_animation  # inlined alongside in the notebook\n\n\ndef _enabled() -> bool:\n    import os\n\n    return os.environ.get("DIGEST", "1").strip() not in {"0", "false", "False"}\n\n\ndef install() -> str:\n    try:\n        from inference.utils import animation as anim_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"digest: SKIP (animation module missing: {exc!r})"\n    try:\n        from inference.framework import solver as solver_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"digest: SKIP (solver module missing: {exc!r})"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"digest: SKIP (tool_agent module missing: {exc!r})"\n\n    for name, mod in (("summarize_animation", anim_mod), ("describe_animation", anim_mod)):\n        if not hasattr(mod, name):\n            return f"digest: SKIP (missing {name})"\n    if not hasattr(solver_mod, "summarize_animation"):\n        return "digest: SKIP (solver lacks summarize_animation binding)"\n    if not hasattr(agent_mod, "describe_animation"):\n        return "digest: SKIP (tool_agent lacks describe_animation binding)"\n    if getattr(anim_mod.summarize_animation, "_digest_patched", False):\n        return "digest: SKIP (already applied)"\n\n    original_summarize = anim_mod.summarize_animation\n    original_describe = anim_mod.describe_animation\n\n    def summarize_with_digest(frames: Any, *, board_changed: bool) -> dict[str, Any] | None:\n        summary = original_summarize(frames, board_changed=board_changed)\n        if summary is None or not _enabled():\n            return summary\n        try:\n            text = digest_animation(frames)\n            if text:\n                summary["digest"] = text\n        except Exception:  # noqa: BLE001 — a broken digest must never break an action\n            pass\n        return summary\n\n    def describe_with_digest(summary: dict[str, Any] | None) -> str:\n        line = original_describe(summary)\n        if not _enabled() or not summary:\n            return line\n        try:\n            text = summary.get("digest")\n            if text:\n                return f"{line} Decoded animation: {text}" if line else f"Decoded animation: {text}"\n        except Exception:  # noqa: BLE001\n            pass\n        return line\n\n    summarize_with_digest._digest_patched = True  # type: ignore[attr-defined]\n    describe_with_digest._digest_patched = True  # type: ignore[attr-defined]\n    # BOTH namespaces, per the silent-no-op lesson\n    anim_mod.summarize_animation = summarize_with_digest\n    solver_mod.summarize_animation = summarize_with_digest\n    anim_mod.describe_animation = describe_with_digest\n    agent_mod.describe_animation = describe_with_digest\n    return "digest: OK"\n'

try:
    import importlib.util as _ilu

    for _name, _src_text in (("digest", _DIGEST_SOURCE), ("graft_digest", _GRAFT_SOURCE)):
        _path = WORKING_DIR / (_name + ".py")
        _path.write_text(_src_text, encoding="utf-8")
        _spec = _ilu.spec_from_file_location(_name, _path)
        _mod = _ilu.module_from_spec(_spec)
        sys.modules[_name] = _mod
        _spec.loader.exec_module(_mod)
    print("[digest]", sys.modules["graft_digest"].install())
except Exception as _exc:  # noqa: BLE001 — fail open to stock
    print(f"[digest] install failed -> stock: {type(_exc).__name__}: {_exc}")


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)